# Resumidor de noticias con NLTK
Kodland M5L3 y proyecto de PNL M6L4.
Código ejecutado localmente con Python; se conserva la salida real. No se ha probado en el servidor de Google Colab. Los extractos compartidos se limitan a 25 palabras por noticia y se conserva su enlace.

In [1]:
import importlib.util, subprocess, sys
for module, package in [('nltk','nltk'),('requests','requests'),('bs4','beautifulsoup4'),('pandas','pandas')]:
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable,'-m','pip','install',package])

## Función summarization(text, n)
Pipeline: frases → palabras → filtro → frecuencias → selección → orden original.

In [2]:
"""Resumen extractivo con NLTK para reducir el tiempo de lectura."""
from collections import Counter
import re
from nltk.tokenize import PunktSentenceTokenizer, wordpunct_tokenize

STOP = set('a al algo ante bajo como con contra cual cuando de del desde donde el ella ellas ellos en entre era es esa ese eso esta este esto fue ha hasta hay la las le les lo los más me mi muy no nos o para pero por porque que qué se ser si sin sobre son su sus también te tu un una unas uno unos y ya'.split())

def summarization(text, n=2):
    """Devuelve hasta n frases importantes, conservando el orden original."""
    if not isinstance(text, str):
        raise TypeError('El texto debe ser una cadena')
    if not isinstance(n, int) or isinstance(n, bool) or n < 1:
        raise ValueError('El número de frases debe ser un entero positivo')
    sentences = PunktSentenceTokenizer().tokenize(text.strip())
    if not sentences:
        return ''
    def tokens(sentence):
        return [w.casefold() for w in wordpunct_tokenize(sentence) if w.isalpha() and w.casefold() not in STOP]
    frequencies = Counter(tokens(text))
    scores = [(i, sum(frequencies[w] for w in tokens(s)) / max(1, len(tokens(s)))) for i, s in enumerate(sentences)]
    selected = sorted(i for i, _ in sorted(scores, key=lambda pair: (-pair[1], pair[0]))[:n])
    return ' '.join(sentences[i] for i in selected)

def resumen_publicable(text, max_words=25):
    """Limita el extracto compartido; el texto íntegro no se almacena."""
    words = summarization(text, 1).split()
    return ' '.join(words[:max_words]) + ('…' if len(words) > max_words else '')



## Pruebas con textos de ejemplo propios

In [3]:
assert summarization('', 2) == ''
assert summarization('Una frase.', 5) == 'Una frase.'
for n in (0, -1):
    try:
        summarization('Texto.', n)
    except ValueError:
        pass
    else:
        raise AssertionError('Debe rechazar longitudes no positivas')
print('Pruebas superadas: texto vacío, texto corto y longitud inválida.')

Pruebas superadas: texto vacío, texto corto y longitud inválida.


## Recopilación de diez noticias
Fuente indicada por la actividad: https://www.xataka.com/tag/ciencia-y-tecnologia
El texto completo se procesa temporalmente; se guardan únicamente el enlace, la fecha, la longitud y el extracto breve.

In [4]:
"""Extraer noticias de la fuente indicada en el curso y resumir diez."""
from concurrent.futures import ThreadPoolExecutor
from urllib.parse import urlparse
import requests
from bs4 import BeautifulSoup
import pandas as pd

SOURCE = 'https://www.xataka.com/tag/ciencia-y-tecnologia'

def scrape_index(url=SOURCE):
    response = requests.get(url, timeout=25)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, 'html.parser')
    urls = []
    for a in soup.select('h2 a[href]'):
        link = a['href']
        if urlparse(link).hostname == 'www.xataka.com' and link not in urls:
            urls.append(link)
    return urls

def scrape_article(item):
    number, url = item
    response = requests.get(url, timeout=25)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, 'html.parser')
    body = soup.select_one('.article-content') or soup.select_one('article')
    if body is None:
        raise ValueError('La página no contiene un artículo reconocible')
    text = ' '.join(p.get_text(' ', strip=True) for p in body.select('p'))
    if len(text) < 100:
        raise ValueError('No se pudo extraer suficiente texto')
    time = soup.select_one('time[datetime]')
    return {'noticia': number, 'url': url,
            'fecha': time.get('datetime', '') if time else '',
            'resumen': resumen_publicable(text, 25),
            'palabras_originales': len(text.split())}

def collect_news(n=10):
    urls = scrape_index()[:n]
    if len(urls) < n:
        raise ValueError(f'Solo se encontraron {len(urls)} enlaces, se necesitan {n}')
    with ThreadPoolExecutor(max_workers=4) as pool:
        rows = list(pool.map(scrape_article, enumerate(urls, 1)))
    df = pd.DataFrame(rows)
    df.to_csv('noticias_resumidas.csv', index=False)
    return df



In [5]:
df = collect_news(10)
assert len(df) == 10
print(df[['noticia','url','fecha','resumen']].to_string(index=False))
print('CSV: noticias_resumidas.csv')

 noticia                                                                                                                                                      url                fecha                                                                                                              resumen
       1 https://www.xataka.com/robotica-e-ia/mayor-repositorio-cientifico-mundo-ha-colapsado-recibir-40-000-pre-prints-este-septiembre-antes-ia-no-pasaba-15-000 2026-10-05T14:45:22Z                                                                                                    ¿Y qué han hecho?
       2                              https://www.xataka.com/robotica-e-ia/mit-ha-disenado-robot-que-vuela-bucea-como-ave-marina-patas-truco-esta-angulo-despegue 2026-07-26T16:30:00Z Es justo lo que acaba de conseguir el MIT : un pequeño robot con alas capaz de volar y sumergirse en el agua , como…
       3                         https://www.xataka.com/investigacion/microscopios-llevaban-casi-sig

## Público objetivo y mejoras
Propuesta: estudiantes que necesitan revisar varios textos en español. Hipótesis de necesidades, sin fingir una encuesta: elegir el número de frases, leer archivos UTF-8, procesar noticias en lote y exportar el resultado. Se implementaron esas funciones; los casos vacíos y los parámetros inválidos producen respuestas claras. Los resúmenes son extractivos y deben revisarse junto con el original.